## Section 1 – LSTM Model Evaluation

The LSTM model was trained in the previous cells. Here we generate predictions on the
test set, inverse-transform both predictions and actuals back to the original scale,
and compute MAE and RMSE so the LSTM can be fairly compared with the classical models.

In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Generate LSTM predictions on the test set
lstm_pred_scaled = model.predict(X_test_lstm)

# Inverse-transform predictions and actuals back to original scale
lstm_pred  = scaler.inverse_transform(lstm_pred_scaled).flatten()
y_test_lstm_actual = scaler.inverse_transform(y_test_lstm.reshape(-1, 1)).flatten()

lstm_mae  = mean_absolute_error(y_test_lstm_actual, lstm_pred)
lstm_rmse = np.sqrt(mean_squared_error(y_test_lstm_actual, lstm_pred))

print(f"LSTM  MAE : {lstm_mae:.4f}")
print(f"LSTM  RMSE: {lstm_rmse:.4f}")


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))
plt.plot(y_test_lstm_actual[:200], label='Actual',          linewidth=1.5)
plt.plot(lstm_pred[:200],          label='LSTM Prediction', linewidth=1.5, linestyle='--')
plt.title('LSTM – Actual vs Predicted Influenza Cases (first 200 test steps)')
plt.xlabel('Time Step')
plt.ylabel('Influenza Cases')
plt.legend()
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()


## Section 2 – XGBoost Model

XGBoost (Extreme Gradient Boosting) is trained on the same feature set and
time-based train/test split used for Linear Regression and Random Forest.
It typically outperforms both on tabular time-series data due to its
regularisation and second-order gradient optimisation.

In [ ]:
# Install xgboost if not already present (Colab has it by default)
try:
    import xgboost as xgb
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'xgboost', '-q'])
    import xgboost as xgb

from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

xgb_model = xgb.XGBRegressor(
    n_estimators   = 300,
    max_depth      = 6,
    learning_rate  = 0.05,
    subsample      = 0.8,
    colsample_bytree = 0.8,
    random_state   = 42,
    verbosity      = 0
)

xgb_model.fit(
    X_train, y_train,
    eval_set   = [(X_test, y_test)],
    verbose    = False
)

xgb_pred = xgb_model.predict(X_test)

xgb_mae  = mean_absolute_error(y_test, xgb_pred)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))

print(f"XGBoost MAE : {xgb_mae:.4f}")
print(f"XGBoost RMSE: {xgb_rmse:.4f}")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Feature importance for XGBoost
xgb_importance = pd.Series(xgb_model.feature_importances_, index=features)
xgb_importance.sort_values().plot(kind='barh', figsize=(8, 5), color='steelblue')
plt.title('Feature Importance – XGBoost')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()


## Section 3 – ARIMA 52-Week Forecast

The ARIMA model was fitted in the preceding notebook. Here we generate a
52-week (one-year) out-of-sample forecast with 95 % confidence intervals
and plot it against the observed historical series.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# 52-week forecast
forecast_result = model_fit.get_forecast(steps=52)
forecast_mean   = forecast_result.predicted_mean
forecast_ci     = forecast_result.conf_int(alpha=0.05)

plt.figure(figsize=(14, 6))
plt.plot(ts[-104:],        label='Historical (last 2 yrs)', color='steelblue')
plt.plot(forecast_mean,    label='ARIMA Forecast',          color='darkorange', linewidth=2)
plt.fill_between(
    forecast_ci.index,
    forecast_ci.iloc[:, 0],
    forecast_ci.iloc[:, 1],
    color='orange', alpha=0.25, label='95% Confidence Interval'
)
plt.title('ARIMA(2,1,2) – 52-Week Influenza Forecast')
plt.xlabel('Date')
plt.ylabel('Influenza Cases (Target)')
plt.legend()
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

print("Forecast summary (first 8 weeks):")
print(pd.DataFrame({'Forecast': forecast_mean,
                    'Lower 95%': forecast_ci.iloc[:, 0],
                    'Upper 95%': forecast_ci.iloc[:, 1]}).head(8).round(2))


## Section 4 – Ensemble (Stacking) Model

A simple stacking ensemble combines the out-of-sample predictions of Linear
Regression, Random Forest, and XGBoost using a Ridge meta-learner. Because we
use TimeSeriesSplit for the first-level predictions, temporal ordering is
strictly preserved and data leakage is avoided.

In [ ]:
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb

features = ['ISO_YEAR', 'ISO_WEEK', 'Month', 'lag_1', 'lag_2', 'roll_3']
df_ml    = df.dropna(subset=features + ['Target'])
X = df_ml[features].values
y = df_ml['Target'].values

tscv = TimeSeriesSplit(n_splits=5)

# ---- Build out-of-fold (OOF) predictions for the meta-learner ----
oof_lr  = np.zeros(len(X))
oof_rf  = np.zeros(len(X))
oof_xgb = np.zeros(len(X))

for train_idx, val_idx in tscv.split(X):
    Xtr, Xval = X[train_idx], X[val_idx]
    ytr        = y[train_idx]

    lr_cv  = LinearRegression().fit(Xtr, ytr)
    rf_cv  = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42).fit(Xtr, ytr)
    xgb_cv = xgb.XGBRegressor(n_estimators=200, max_depth=5, learning_rate=0.05,
                               random_state=42, verbosity=0).fit(Xtr, ytr)

    oof_lr [val_idx] = lr_cv.predict(Xval)
    oof_rf [val_idx] = rf_cv.predict(Xval)
    oof_xgb[val_idx] = xgb_cv.predict(Xval)

# ---- Train meta-learner on OOF predictions ----
split     = int(len(X) * 0.8)
meta_X_train = np.column_stack([oof_lr[:split],  oof_rf[:split],  oof_xgb[:split]])
meta_X_test  = np.column_stack([oof_lr[split:],  oof_rf[split:],  oof_xgb[split:]])
meta_y_train = y[:split]
meta_y_test  = y[split:]

meta_model = Ridge(alpha=1.0)
meta_model.fit(meta_X_train, meta_y_train)
ensemble_pred = meta_model.predict(meta_X_test)

ens_mae  = mean_absolute_error(meta_y_test, ensemble_pred)
ens_rmse = np.sqrt(mean_squared_error(meta_y_test, ensemble_pred))

print(f"Ensemble (Stacking) MAE : {ens_mae:.4f}")
print(f"Ensemble (Stacking) RMSE: {ens_rmse:.4f}")
print(f"Meta-learner weights – LR: {meta_model.coef_[0]:.4f}  "
      f"RF: {meta_model.coef_[1]:.4f}  XGB: {meta_model.coef_[2]:.4f}")


## Section 5 – Hyperparameter Tuning (Random Forest)

RandomizedSearchCV with TimeSeriesSplit is used to tune the Random Forest.
Using TimeSeriesSplit instead of the default k-fold ensures that future
observations never leak into the training fold during cross-validation.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor
import numpy as np
import pandas as pd

features = ['ISO_YEAR', 'ISO_WEEK', 'Month', 'lag_1', 'lag_2', 'roll_3']
df_ml    = df.dropna(subset=features + ['Target'])
X_all    = df_ml[features]
y_all    = df_ml['Target']

split    = int(len(df_ml) * 0.8)
X_train_ = X_all.iloc[:split]
y_train_ = y_all.iloc[:split]
X_test_  = X_all.iloc[split:]
y_test_  = y_all.iloc[split:]

param_grid = {
    'n_estimators' : [100, 200, 300],
    'max_depth'    : [5, 8, 12, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf' : [1, 2, 4],
    'max_features' : ['sqrt', 'log2', 0.8]
}

tscv = TimeSeriesSplit(n_splits=5)

rs = RandomizedSearchCV(
    RandomForestRegressor(random_state=42),
    param_distributions = param_grid,
    n_iter              = 30,
    scoring             = 'neg_mean_absolute_error',
    cv                  = tscv,
    random_state        = 42,
    n_jobs              = -1,
    verbose             = 1
)
rs.fit(X_train_, y_train_)

best_rf   = rs.best_estimator_
best_pred = best_rf.predict(X_test_)

print("\nBest parameters:", rs.best_params_)
print(f"Tuned RF  MAE : {mean_absolute_error(y_test_, best_pred):.4f}")
print(f"Tuned RF  RMSE: {np.sqrt(mean_squared_error(y_test_, best_pred)):.4f}")


## Section 6 – Comprehensive Model Comparison

All five models are evaluated on the shared test set using MAE, RMSE, and R².
The bar charts use a consistent colour scheme to make the trade-offs immediately
visible.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ── Gather metrics for all models ──────────────────────────────────────────
split    = int(len(df_ml) * 0.8)
y_test_common = df_ml['Target'].iloc[split:].values

def safe_metrics(y_true, y_hat, label):
    mae  = mean_absolute_error(y_true, y_hat)
    rmse = np.sqrt(mean_squared_error(y_true, y_hat))
    r2   = r2_score(y_true, y_hat)
    return {'Model': label, 'MAE': mae, 'RMSE': rmse, 'R²': r2}

rows = [
    safe_metrics(y_test_common, lr_pred,       'Linear Regression'),
    safe_metrics(y_test_common, rf_pred,        'Random Forest'),
    safe_metrics(y_test_common, xgb_pred,       'XGBoost'),
    safe_metrics(y_test_common, best_pred,      'RF (Tuned)'),
    safe_metrics(meta_y_test,   ensemble_pred,  'Stacking Ensemble'),
]

# LSTM uses a different test window; add separately
rows.append({'Model': 'LSTM', 'MAE': lstm_mae, 'RMSE': lstm_rmse, 'R²': float('nan')})

comparison = pd.DataFrame(rows).set_index('Model').round(4)
print(comparison.to_string())


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
palette   = sns.color_palette('tab10', n_colors=len(comparison))

for ax, metric in zip(axes, ['MAE', 'RMSE', 'R²']):
    vals = comparison[metric].dropna()
    bars = ax.bar(vals.index, vals.values, color=palette[:len(vals)])
    ax.set_title(f'{metric} Comparison', fontsize=13)
    ax.set_ylabel(metric)
    ax.set_xticklabels(vals.index, rotation=30, ha='right')
    for bar, val in zip(bars, vals.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f'{val:.2f}', ha='center', va='bottom', fontsize=9)
    ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.suptitle('All-Model Performance Comparison', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()


## Section 7 – Saving All Trained Models

All models are serialised to `/content/drive/MyDrive/influenza_models/` so they
can be loaded later for inference without re-training. The scaler is also saved
because it is required to inverse-transform LSTM predictions at serve time.

In [ ]:
import os, joblib

save_dir = '/content/drive/MyDrive/influenza_models/'
os.makedirs(save_dir, exist_ok=True)

# Classical ML models
joblib.dump(lr,          save_dir + 'linear_regression.pkl')
joblib.dump(rf,          save_dir + 'random_forest.pkl')
joblib.dump(best_rf,     save_dir + 'random_forest_tuned.pkl')
joblib.dump(xgb_model,   save_dir + 'xgboost_model.pkl')
joblib.dump(meta_model,  save_dir + 'stacking_meta_learner.pkl')
joblib.dump(scaler,      save_dir + 'minmax_scaler.pkl')

# ARIMA
model_fit.save(save_dir + 'arima_model.pkl')

# LSTM (Keras / TensorFlow)
model.save(save_dir + 'lstm_model.keras')

print("All models saved to:", save_dir)
print(os.listdir(save_dir))


## Section 8 – Reusable Prediction Pipeline

`predict_influenza()` is the single inference entry-point.  
Given a country name, an ISO year, and an ISO week it:
1. Reconstructs the required lag and rolling features from the stored dataset.
2. Returns predictions from all models in a ranked summary table.

This function is the backbone of the Streamlit deployment in Section 9.

In [ ]:
import pandas as pd
import numpy as np
import joblib

# ── Load models (safe to call even if already in memory) ────────────────────
_LR     = joblib.load(save_dir + 'linear_regression.pkl')
_RF     = joblib.load(save_dir + 'random_forest_tuned.pkl')
_XGB    = joblib.load(save_dir + 'xgboost_model.pkl')
_SCALER = joblib.load(save_dir + 'minmax_scaler.pkl')


def _build_features(country: str, iso_year: int, iso_week: int,
                    df_ref: pd.DataFrame) -> pd.DataFrame:
    """Reconstruct lag/rolling features for a single prediction point."""
    hist = (
        df_ref[df_ref['COUNTRY_AREA_TERRITORY'] == country]
        .sort_values(['ISO_YEAR', 'ISO_WEEK'])
        [['ISO_YEAR', 'ISO_WEEK', 'Target']]
        .tail(10)           # only the last 10 weeks needed
    )

    if len(hist) < 3:
        raise ValueError(f"Not enough history for country '{country}'.")

    lag_1  = hist['Target'].iloc[-1]
    lag_2  = hist['Target'].iloc[-2]
    lag_3  = hist['Target'].iloc[-3]
    roll_3 = hist['Target'].iloc[-3:].mean()
    month  = ((iso_week - 1) // 4) + 1

    return pd.DataFrame([{
        'ISO_YEAR': iso_year,
        'ISO_WEEK': iso_week,
        'Month'   : month,
        'lag_1'   : lag_1,
        'lag_2'   : lag_2,
        'roll_3'  : roll_3
    }])


def predict_influenza(country: str, iso_year: int, iso_week: int,
                      df_ref: pd.DataFrame = None) -> pd.DataFrame:
    """
    Predict influenza cases for a given country / year / week.

    Parameters
    ----------
    country  : str  – e.g. 'India'
    iso_year : int  – e.g. 2025
    iso_week : int  – 1–52
    df_ref   : pd.DataFrame – the cleaned WHO dataframe (defaults to global `df`)

    Returns
    -------
    pd.DataFrame with one row per model showing the predicted case count.
    """
    if df_ref is None:
        df_ref = df          # fall back to the globally loaded dataframe

    feat_df = _build_features(country, iso_year, iso_week, df_ref)
    feat_cols = ['ISO_YEAR', 'ISO_WEEK', 'Month', 'lag_1', 'lag_2', 'roll_3']

    preds = {
        'Linear Regression': float(_LR.predict(feat_df[feat_cols])[0]),
        'Random Forest (Tuned)': float(_RF.predict(feat_df[feat_cols])[0]),
        'XGBoost'           : float(_XGB.predict(feat_df[feat_cols])[0]),
    }

    # Clip negative predictions to zero (physically meaningless)
    preds = {k: max(0.0, v) for k, v in preds.items()}

    result = pd.DataFrame.from_dict(
        preds, orient='index', columns=['Predicted Cases']
    ).round(2)
    result.index.name = 'Model'
    return result


# ── Quick sanity check ───────────────────────────────────────────────────────
print(predict_influenza('India', 2024, 10))


## Section 9 – Streamlit Deployment App

Run the cell below to write `influenza_app.py` to disk, then launch it
with `!streamlit run influenza_app.py &` (in Colab use `localtunnel` to
expose the port, or deploy to [Streamlit Community Cloud](https://streamlit.io/cloud)
by pushing this file plus a `requirements.txt` to a GitHub repo).

In [ ]:
app_code = '''
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import os

# ─── Page config ─────────────────────────────────────────────────────────────
st.set_page_config(
    page_title  = "Seasonal Influenza Predictor",
    page_icon   = "🦠",
    layout      = "wide",
)

st.title("🦠 Seasonal Influenza Prediction Dashboard")
st.markdown(
    "Predict weekly influenza case counts using WHO surveillance data "
    "and machine learning models."
)

# ─── Load artefacts ──────────────────────────────────────────────────────────
MODEL_DIR = os.environ.get("MODEL_DIR", "./influenza_models/")
DATA_PATH = os.environ.get("DATA_PATH", "./Influenza_Dataset.csv")

@st.cache_resource
def load_models():
    lr     = joblib.load(MODEL_DIR + "linear_regression.pkl")
    rf     = joblib.load(MODEL_DIR + "random_forest_tuned.pkl")
    xgb_m  = joblib.load(MODEL_DIR + "xgboost_model.pkl")
    scaler = joblib.load(MODEL_DIR + "minmax_scaler.pkl")
    return lr, rf, xgb_m, scaler

@st.cache_data
def load_data():
    df = pd.read_csv(DATA_PATH, low_memory=False)
    df["Target"] = df["INF_A"].fillna(0) + df["INF_B"].fillna(0)
    df["ISO_YEAR"] = pd.to_numeric(df["ISO_YEAR"], errors="coerce")
    df["ISO_WEEK"] = pd.to_numeric(df["ISO_WEEK"], errors="coerce")
    df["Month"]    = ((df["ISO_WEEK"] - 1) // 4) + 1
    df["lag_1"]    = df.groupby("COUNTRY_AREA_TERRITORY")["Target"].shift(1)
    df["lag_2"]    = df.groupby("COUNTRY_AREA_TERRITORY")["Target"].shift(2)
    df["roll_3"]   = (df.groupby("COUNTRY_AREA_TERRITORY")["Target"]
                       .rolling(3).mean().reset_index(0, drop=True))
    df.fillna(0, inplace=True)
    return df

try:
    lr_m, rf_m, xgb_m, scaler = load_models()
    df = load_data()
    models_loaded = True
except Exception as e:
    st.error(f"Could not load models or data: {e}")
    models_loaded = False

# ─── Sidebar – inputs ────────────────────────────────────────────────────────
st.sidebar.header("📋 Prediction Inputs")

if models_loaded:
    countries = sorted(df["COUNTRY_AREA_TERRITORY"].dropna().unique())
    country   = st.sidebar.selectbox("Country", countries,
                                     index=countries.index("India") if "India" in countries else 0)
    iso_year  = st.sidebar.slider("ISO Year",  2015, 2026, 2024)
    iso_week  = st.sidebar.slider("ISO Week",  1,    52,   10)

    # ── Build features ──────────────────────────────────────────────────────
    hist = (df[df["COUNTRY_AREA_TERRITORY"] == country]
            .sort_values(["ISO_YEAR", "ISO_WEEK"])
            [["ISO_YEAR", "ISO_WEEK", "Target"]]
            .tail(10))

    if len(hist) >= 3:
        lag_1  = hist["Target"].iloc[-1]
        lag_2  = hist["Target"].iloc[-2]
        roll_3 = hist["Target"].iloc[-3:].mean()
        month  = ((iso_week - 1) // 4) + 1

        feat = pd.DataFrame([{
            "ISO_YEAR": iso_year, "ISO_WEEK": iso_week,
            "Month": month,       "lag_1": lag_1,
            "lag_2": lag_2,       "roll_3": roll_3
        }])

        feat_cols = ["ISO_YEAR","ISO_WEEK","Month","lag_1","lag_2","roll_3"]

        preds = {
            "Linear Regression"    : float(lr_m.predict(feat[feat_cols])[0]),
            "Random Forest (Tuned)": float(rf_m.predict(feat[feat_cols])[0]),
            "XGBoost"              : float(xgb_m.predict(feat[feat_cols])[0]),
        }
        preds = {k: max(0.0, round(v, 2)) for k, v in preds.items()}

        # ── Main panel ──────────────────────────────────────────────────────
        st.subheader(f"📍 Prediction for {country} | Year {iso_year} | Week {iso_week}")

        col1, col2, col3 = st.columns(3)
        col1.metric("Linear Regression",     f"{preds['Linear Regression']:.1f} cases")
        col2.metric("Random Forest (Tuned)", f"{preds['Random Forest (Tuned)']:.1f} cases")
        col3.metric("XGBoost",               f"{preds['XGBoost']:.1f} cases")

        # Bar chart of predictions
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.bar(preds.keys(), preds.values(), color=["steelblue","seagreen","darkorange"])
        ax.set_ylabel("Predicted Cases")
        ax.set_title("Model Predictions Comparison")
        ax.grid(axis="y", alpha=0.4)
        st.pyplot(fig)

        # ── Historical trend for this country ───────────────────────────────
        st.subheader(f"📈 Historical Weekly Trend – {country}")
        country_hist = (df[df["COUNTRY_AREA_TERRITORY"] == country]
                        .sort_values(["ISO_YEAR", "ISO_WEEK"])
                        .tail(104))

        fig2, ax2 = plt.subplots(figsize=(12, 4))
        ax2.plot(country_hist["Target"].values, color="steelblue", linewidth=1.5)
        ax2.set_title(f"Last 2 Years – Weekly Influenza Cases ({country})")
        ax2.set_xlabel("Weeks (most recent on right)")
        ax2.set_ylabel("Cases (Inf A + Inf B)")
        ax2.grid(alpha=0.4)
        st.pyplot(fig2)

    else:
        st.warning(f"Not enough historical data for '{country}' to compute lag features.")

# ─── Footer ──────────────────────────────────────────────────────────────────
st.markdown("---")
st.caption(
    "Data source: WHO Global Influenza Surveillance and Response System (GISRS). "
    "Predictions are trend estimates and should not be used as clinical diagnoses."
)
'''

with open('influenza_app.py', 'w') as f:
    f.write(app_code)

print("influenza_app.py written successfully.")
print("\nTo launch locally:")
print("  !pip install streamlit -q")
print("  !streamlit run influenza_app.py &")
print("\nTo expose via localtunnel in Colab:")
print("  !npm install -g localtunnel -q")
print("  !npx localtunnel --port 8501 &")


## Section 10 – requirements.txt for Deployment

Generates a `requirements.txt` pinned to the library versions used in Colab.
Push this alongside `influenza_app.py` and your saved models to GitHub, then
connect the repo to **Streamlit Community Cloud** for a free hosted deployment.

In [ ]:
req = """pandas>=1.5
numpy>=1.23
scikit-learn>=1.2
xgboost>=1.7
statsmodels>=0.13
tensorflow>=2.12
matplotlib>=3.6
seaborn>=0.12
streamlit>=1.32
joblib>=1.2
"""

with open('requirements.txt', 'w') as f:
    f.write(req)

print("requirements.txt written:")
print(req)


## Section 11 – Deployment Guide

### Step-by-step: Streamlit Community Cloud (free)

1. **Create a GitHub repository** (e.g. `influenza-predictor`).
2. **Upload these files** to the repo root:
   - `influenza_app.py`
   - `requirements.txt`
   - `influenza_models/` folder (all `.pkl` and `.keras` files)
   - `Influenza_Dataset.csv`
3. **Go to** [share.streamlit.io](https://share.streamlit.io), sign in with GitHub.
4. Click **New app → select your repo → set main file to `influenza_app.py`**.
5. Add two **Secrets** (optional) if you store models/data elsewhere:
   ```
   MODEL_DIR = "./influenza_models/"
   DATA_PATH = "./Influenza_Dataset.csv"
   ```
6. Click **Deploy** – Streamlit installs from `requirements.txt` automatically.

### For larger models (Google Drive → HuggingFace Hub)
If your LSTM `.keras` file is too large for GitHub (>100 MB), upload it to
[HuggingFace Hub](https://huggingface.co/models) and fetch it in `influenza_app.py`
using `huggingface_hub.hf_hub_download()`.

---

### Local testing (Colab)

```python
!pip install streamlit pyngrok -q
from pyngrok import ngrok
!streamlit run influenza_app.py &
public_url = ngrok.connect(8501)
print("App URL:", public_url)
```
